# 📝 Week 5: Keyword Extraction pada Dataset MBG

Notebook ini membahas proses ekstraksi kata kunci dari kumpulan artikel berita Makanan Bergizi Gratis (MBG) menggunakan pendekatan **TF-IDF** dan **N-gram Analysis**.

## 🎯 Tujuan Pembelajaran:
1. Memuat dan memahami struktur dataset artikel MBG
2. Melakukan preprocessing teks bahasa Indonesia
3. Mengekstrak kata kunci menggunakan TF-IDF
4. Menganalisis pola frasa penting menggunakan N-gram (bigram & trigram)

## 📦 Instalasi Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

%pip install -q nltk scikit-learn pandas openpyxl

print("✅ Semua library berhasil diinstall!")

## 📚 Import Libraries

In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ Import libraries berhasil!")

---

## 1️⃣ Load Dataset

Dataset berisi artikel berita tentang program **Makanan Bergizi Gratis (MBG)** yang dikumpulkan dari berbagai media Indonesia.

File Excel terdiri dari dua sheet:
- **`Case`** — artikel kasus/insiden terkait MBG
- **`Main`** — artikel utama tentang program MBG

| Kolom | Deskripsi |
|---|---|
| ID | Nomor unik artikel |
| Judul | Judul berita |
| Konten | Isi artikel lengkap |
| Kategori | Kategori berita |
| Sumber | Media online sumber |
| Tanggal | Tanggal terbit |
| Tone | Nada berita (positif/negatif/netral) |

In [ ]:
# Load dataset dari file Excel
file_path = 'artikel_manual_celine.xlsx'

df_case = pd.read_excel(file_path, sheet_name='Case')
df_main = pd.read_excel(file_path, sheet_name='Main')

print("✅ Dataset berhasil dimuat!")
print(f"Sheet 'Case'  : {df_case.shape[0]} artikel, {df_case.shape[1]} kolom")
print(f"Sheet 'Main'  : {df_main.shape[0]} artikel, {df_main.shape[1]} kolom")

In [ ]:
# Preview sheet Main
df_main.head(3)

In [ ]:
# Preview sheet Case
df_case.head(3)

---

## 2️⃣ Preprocessing Teks

Sebelum ekstraksi kata kunci, teks perlu diproses agar lebih bersih dan terstandarisasi:

### 2.1 Fungsi Preprocessing

In [ ]:
# Stopwords bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))

def preprocess_text(text):
    """Lowercase, hapus tanda baca & angka, tokenisasi, hapus stopwords."""
    if pd.isna(text):
        return ''
    # Lowercase
    text = text.lower()
    # Hapus tanda baca dan angka
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenisasi
    tokens = word_tokenize(text)
    # Hapus stopwords
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)

print("✅ Fungsi preprocessing siap!")

### 2.2 Terapkan Preprocessing

In [ ]:
# Terapkan preprocessing ke kolom Konten
df_main['Konten_Clean'] = df_main['Konten'].apply(preprocess_text)
df_case['Konten_Clean'] = df_case['Konten'].apply(preprocess_text)

print("✅ Preprocessing selesai!")
print("\nContoh teks asli (30 kata pertama):")
print(df_main['Konten'].iloc[0][:200])
print("\nContoh teks setelah preprocessing (30 token):")
print(' '.join(df_main['Konten_Clean'].iloc[0].split()[:30]))

---

## 3️⃣ Ekstraksi Kata Kunci dengan TF-IDF

**TF-IDF** (Term Frequency–Inverse Document Frequency) mengukur seberapa penting suatu kata dalam satu dokumen relatif terhadap seluruh koleksi dokumen.

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}$$

### 3.1 TF-IDF pada Sheet Main

In [ ]:
# TF-IDF vectorizer
documents = df_main['Konten'].astype(str).tolist()

vectorizer = TfidfVectorizer(
    max_features=20,
    ngram_range=(1, 2),
    stop_words=list(stop_words)
)
tfidf_matrix = vectorizer.fit_transform(documents)

# Buat DataFrame hasil TF-IDF
feature_names = vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.toarray()
tfidf_df = pd.DataFrame(tfidf_scores, columns=feature_names)

# Tampilkan top kata kunci berdasarkan total skor
top_keywords = tfidf_df.sum().sort_values(ascending=False).head(20)
print("✅ TF-IDF selesai dihitung!")
print("\n🔑 Top 20 Kata Kunci (Sheet Main):")
print(top_keywords.to_string())

### 3.2 TF-IDF pada Sheet Case

In [ ]:
# TF-IDF untuk sheet Case
documents_case = df_case['Konten'].astype(str).tolist()

vectorizer_case = TfidfVectorizer(
    max_features=20,
    ngram_range=(1, 2),
    stop_words=list(stop_words)
)
tfidf_matrix_case = vectorizer_case.fit_transform(documents_case)

feature_names_case = vectorizer_case.get_feature_names_out()
tfidf_scores_case = tfidf_matrix_case.toarray()
tfidf_df_case = pd.DataFrame(tfidf_scores_case, columns=feature_names_case)

top_keywords_case = tfidf_df_case.sum().sort_values(ascending=False).head(20)
print("✅ TF-IDF Sheet Case selesai!")
print("\n🔑 Top 20 Kata Kunci (Sheet Case):")
print(top_keywords_case.to_string())

### 3.3 Perbandingan Top Keywords Antar Sheet

In [ ]:
# Bandingkan top keywords dari kedua sheet
comparison = pd.DataFrame({
    'Kata Kunci (Main)': top_keywords.index.tolist()[:10],
    'Skor Main': top_keywords.values[:10].round(4),
    'Kata Kunci (Case)': top_keywords_case.index.tolist()[:10],
    'Skor Case': top_keywords_case.values[:10].round(4)
})

print("✅ Perbandingan top 10 kata kunci:")
print(comparison.to_string(index=False))

---

## 4️⃣ Analisis N-gram

N-gram membantu mengidentifikasi frasa penting yang terdiri dari beberapa kata yang sering muncul bersama-sama.

- **Bigram** = 2 kata berurutan → contoh: *"ibu hamil"*, *"gizi buruk"*
- **Trigram** = 3 kata berurutan → contoh: *"program makan gratis"*

### 4.1 Bigram dan Trigram pada Sheet Main

In [ ]:
# CountVectorizer untuk bigram dan trigram
texts_main = df_main['Konten'].dropna().tolist()

ngram_vectorizer = CountVectorizer(
    ngram_range=(2, 3),
    stop_words=list(stop_words),
    max_features=15
)
X_ngram = ngram_vectorizer.fit_transform(texts_main)

ngrams = ngram_vectorizer.get_feature_names_out()
counts = X_ngram.sum(axis=0).A1
ngram_freq = pd.DataFrame({'ngram': ngrams, 'count': counts}).sort_values(by='count', ascending=False)

print("✅ N-gram analisis selesai!")
print("\n📊 Top Bigram & Trigram (Sheet Main):")
print(ngram_freq.to_string(index=False))

### 4.2 Bigram dan Trigram pada Sheet Case

In [ ]:
# N-gram untuk sheet Case
texts_case = df_case['Konten'].dropna().tolist()

ngram_vectorizer_case = CountVectorizer(
    ngram_range=(2, 3),
    stop_words=list(stop_words),
    max_features=15
)
X_ngram_case = ngram_vectorizer_case.fit_transform(texts_case)

ngrams_case = ngram_vectorizer_case.get_feature_names_out()
counts_case = X_ngram_case.sum(axis=0).A1
ngram_freq_case = pd.DataFrame({'ngram': ngrams_case, 'count': counts_case}).sort_values(by='count', ascending=False)

print("✅ N-gram Sheet Case selesai!")
print("\n📊 Top Bigram & Trigram (Sheet Case):")
print(ngram_freq_case.to_string(index=False))

### 4.3 Export Hasil ke Excel

In [ ]:
# Export hasil ke Excel
output_file = 'keyword_extraction_results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    top_keywords.reset_index().rename(columns={'index': 'keyword', 0: 'score'}).to_excel(
        writer, sheet_name='TFIDF_Main', index=False
    )
    top_keywords_case.reset_index().rename(columns={'index': 'keyword', 0: 'score'}).to_excel(
        writer, sheet_name='TFIDF_Case', index=False
    )
    ngram_freq.to_excel(writer, sheet_name='Ngram_Main', index=False)
    ngram_freq_case.to_excel(writer, sheet_name='Ngram_Case', index=False)

print(f"✅ Hasil diekspor ke '{output_file}'")

---

## 📝 Kesimpulan

- 🎯 **TF-IDF** berhasil mengidentifikasi kata kunci dominan dari artikel MBG; kata-kata seperti *program*, *mbg*, *makanan*, *anak*, dan *gizi* muncul paling signifikan.
- 📊 **Sheet Case** (artikel insiden) menampilkan kata kunci berbeda, seperti *racun*, *basi*, *keracunan* — mencerminkan isu keamanan pangan.
- 🔧 **N-gram** (bigram/trigram) membantu mengungkap frasa penting seperti *"ibu hamil"*, *"gizi buruk"*, *"program makan gratis"* yang tidak terlihat dari analisis unigram saja.
- 📂 Seluruh hasil diekspor ke file Excel untuk analisis lebih lanjut.

---

## 📚 Referensi

- Salton, G., & Buckley, C. (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management*, 24(5), 513–523.
- Scikit-learn Documentation: [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- NLTK Documentation: [stopwords](https://www.nltk.org/)

---

**Terima kasih! Happy Learning! 🎓**